In [5]:
import gc
import glob
from pathlib import Path

import numpy as np
from scipy.stats import pearsonr, fisher_exact, spearmanr,
from statsmodels.stats.multitest import multipletests

import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [18]:
EQTL_PATH = '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists'
CSV_PATH = '/mnt/sdb/scz_meta_analysis_processed/dge_signatures/dreamlet_dges/disease_analyses/'

In [20]:
## FDR Calculation Function ##

def fdr_per_celltype(dge_df):
    """Add FDR-adjusted p-values and significance per cell type."""
    dge_df['p.adjusted'] = None
    dge_df['significant_fdr'] = False
    
    for celltype, group in dge_df.groupby('assay'):
        rejected, pvals_corrected, _, _ = multipletests(group['p.value'], method='fdr_bh')
        dge_df.loc[group.index, 'p.adjusted'] = pvals_corrected
        dge_df.loc[group.index, 'significant_fdr'] = rejected
    
    return dge_df

In [48]:
## Import Prasanth DF ##

prasanth_annot = pd.read_csv('/home/deepak/datasets/annotations/all_genes_EnsemblInfo_104.csv')

/tmp/ipykernel_188148/3672267452.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  prasanth_annot = pd.read_csv('/home/deepak/datasets/annotations/all_genes_EnsemblInfo_104.csv')


In [53]:
## Read in Fetal EQTL Papers eQTLs-eGenes ##

# https://pmc.ncbi.nlm.nih.gov/articles/PMC6231252/

fetal_eqtl_df = pd.read_excel(Path(EQTL_PATH) / 'Fetal_eQTL_13059_2018_1567_MOESM1_ESM.xlsx', sheet_name='TblS6', skiprows=2)

# https://springernature.figshare.com/articles/dataset/
# Summary_statistics_for_expression_quantitative_trait_loci_in_the_developing_human_brain_and_their_enrichment_in_
# neuropsychiatric_disorders/6881825
# all eqtls not just significant ones
fetal_all_eqtl_df = pd.read_table(Path(EQTL_PATH) / 'all_eqtls_gene.txt.gz', compression='gzip')

fetal_all_eqtl_df = fetal_all_eqtl_df.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

In [120]:
fetal_eqtl_df

,Chromosome,Position,rsID,ENSEMBL_geneID,SYMBOL,TSS_distance,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,minor_allele_samples,minor_allele_count,maf,pval_nominal,slope,slope_se,pval_perm,pval_beta,qval
0,chr1,39901953,rs11207097,ENSG00000117013,KCNQ4,-882060,4115,1.04907,328.1710,84.1406,3.749370e-06,15,16,0.066667,4.554440e-07,0.566137,0.104885,0.0010,0.000866,0.019243
1,chr1,150537646,rs11810419,ENSG00000143401,ANP32E,-301491,1891,1.04346,71.7949,81.3086,2.132060e-05,24,26,0.108333,2.398570e-06,0.369258,0.073776,0.0007,0.001133,0.024079
2,chr1,154680797,rs10908424,ENSG00000160710,ADAR,-52799,2538,1.04032,205.1570,84.0039,1.073740e-07,71,82,0.341667,6.617400e-09,-0.656275,0.103475,0.0001,0.000014,0.000536
3,chr1,247937246,rs61857507,ENSG00000177233,OR2M1P,-184687,4584,1.06809,281.7300,82.0117,5.988330e-06,79,100,0.416667,5.693260e-07,-0.565319,0.105750,0.0009,0.001059,0.022871
4,chr1,57886800,rs1202826,ENSG00000184292,TACSTD2,690972,4383,1.03269,379.6230,86.2109,3.835200e-06,43,52,0.216667,6.429380e-07,-0.536713,0.100935,0.0012,0.001159,0.024434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,chr21,43417671,rs430552,ENSG00000142178,SIK1,9456,5020,1.06259,463.4940,84.1406,8.592730e-09,58,70,0.291667,3.421750e-10,0.618027,0.088652,0.0001,0.000002,0.000087
168,chr21,45284264,rs59081765,ENSG00000160294,MCM3AP,1002032,4005,1.06212,257.5560,82.0508,6.129920e-06,13,13,0.054167,5.893210e-07,-0.824770,0.154517,0.0003,0.001028,0.022315
169,chr21,39101271,rs724732,ENSG00000223806,LINC00114,-353812,4955,1.04043,511.6320,86.3867,3.947760e-06,22,25,0.104167,6.824480e-07,-0.786307,0.148262,0.0009,0.001543,0.030811
170,chr21,41111102,rs56767850,ENSG00000226496,LINC00323,37030,5930,1.03839,872.9380,88.5742,1.026800e-06,38,41,0.170833,2.061790e-07,-0.505511,0.090594,0.0008,0.000673,0.015529


In [12]:
## Read in Raj Towfique EQTL Papers eQTLs-eGenes ##

# https://www.nature.com/articles/s41588-026-02541-x

scz_eqtl_df = pd.read_excel(Path(EQTL_PATH) / 'Towfique_SCZ_EQTL_41588_2026_2541_MOESM4_ESM.xlsx', sheet_name='Table11_COLOC', skiprows=2)

In [107]:
# Read in SingleBrain eQTLs used in Raj's paper #

files = [
    '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists/SingleBrain_eQTL_files/OPC_eqtl_top_assoc.tsv.gz',
    '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists/SingleBrain_eQTL_files/IN_eqtl_top_assoc.tsv.gz',
    '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists/SingleBrain_eQTL_files/MG_eqtl_top_assoc.tsv.gz',
    '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists/SingleBrain_eQTL_files/OD_eqtl_top_assoc.tsv.gz',
    '/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/eqtl_lists/SingleBrain_eQTL_files/End_eqtl_top_assoc.tsv.gz'
]

opc_singlebrain_eqtl = pd.read_csv(files[0], sep='\t')
in_singlebrain_eqtl = pd.read_csv(files[1], sep='\t')
mg_singlebrain_eqtl = pd.read_csv(files[2], sep='\t')
od_singlebrain_eqtl = pd.read_csv(files[3], sep='\t')
end_singlebrain_eqtl = pd.read_csv(files[4], sep='\t')

# extract true ensembls without dot
opc_singlebrain_eqtl['gene_id'] = opc_singlebrain_eqtl['feature'].str.split('.').str[0]
in_singlebrain_eqtl['gene_id'] = in_singlebrain_eqtl['feature'].str.split('.').str[0]
mg_singlebrain_eqtl['gene_id'] = mg_singlebrain_eqtl['feature'].str.split('.').str[0]
od_singlebrain_eqtl['gene_id'] = od_singlebrain_eqtl['feature'].str.split('.').str[0]
end_singlebrain_eqtl['gene_id'] = end_singlebrain_eqtl['feature'].str.split('.').str[0]

# merge on hgnc ids
opc_singlebrain_eqtl = opc_singlebrain_eqtl.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

in_singlebrain_eqtl = in_singlebrain_eqtl.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

mg_singlebrain_eqtl = mg_singlebrain_eqtl.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

od_singlebrain_eqtl = od_singlebrain_eqtl.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

end_singlebrain_eqtl = end_singlebrain_eqtl.merge(prasanth_annot[['ensembl_gene_id', 'hgnc_symbol']], 
                        left_on='gene_id', right_on='ensembl_gene_id')

In [27]:
## Single Cell ##

dge_3q29 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_purcell_dge_scz_results.csv')
dge_15q13 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_walsh_15q13_dge_scz_results.csv')
dge_nrxn1 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_nrxn1_combined.csv')
dge_22q11 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_22q11_combined.csv')
dge_idiopathic = pd.read_csv(Path(CSV_PATH) / 'dreamlet_idiopathic_combined.csv')

# fdr correct meta-estimates
dge_idiopathic = fdr_per_celltype(dge_idiopathic)
dge_22q11 = fdr_per_celltype(dge_22q11)
dge_nrxn1 = fdr_per_celltype(dge_nrxn1)

# align with meta-estimates
dge_3q29['estimate'] = dge_3q29['logFC']
dge_3q29['significant_fdr'] = (dge_3q29['adj.P.Val'] < 0.05)
dge_15q13['estimate'] = dge_15q13['logFC']
dge_15q13['significant_fdr'] = (dge_15q13['adj.P.Val'] < 0.05)

## Bulk ##

dge_3q29_bulk = pd.read_csv(Path(CSV_PATH) / 'dreamlet_purcell_dge_scz_bulk_results.csv')
dge_15q13_bulk  = pd.read_csv(Path(CSV_PATH) / 'dreamlet_walsh_15q13_dge_scz_bulk_results.csv')
dge_nrxn1_bulk  = pd.read_csv(Path(CSV_PATH) / 'dreamlet_nrxn1_combined_bulk.csv')
dge_22q11_bulk  = pd.read_csv(Path(CSV_PATH) / 'dreamlet_22q11_combined_bulk.csv')
dge_idiopathic_bulk  = pd.read_csv(Path(CSV_PATH) / 'dreamlet_idiopathic_combined_bulk.csv')

# fdr correct bulk meta-estimates
dge_idiopathic_bulk = fdr_per_celltype(dge_idiopathic_bulk)
dge_22q11_bulk = fdr_per_celltype(dge_22q11_bulk)
dge_nrxn1_bulk = fdr_per_celltype(dge_nrxn1_bulk)

# align with meta-estimates
dge_3q29_bulk['estimate'] = dge_3q29_bulk['logFC']
dge_3q29_bulk['significant_fdr'] = (dge_3q29_bulk['adj.P.Val'] < 0.05)
dge_15q13_bulk['estimate'] = dge_15q13_bulk['logFC']
dge_15q13_bulk['significant_fdr'] = (dge_15q13_bulk['adj.P.Val'] < 0.05)

In [121]:
## This will test all your DGEs across celltypes where if a DGE is significnat in any celltype, you call it significant ##

In [75]:
def test_eqtl_enrichment(
    dge_df,
    eqtl_sig_df,
    eqtl_all_df,
    gene_col="ID",
    sig_col="significant_fdr",
    eqtl_gene_col="SYMBOL",
    eqtl_all_gene_col="hgnc_symbol",
    alternative="greater",
):
    """
    Fisher exact test for enrichment of significant DGE genes among significant eGenes.

    Parameters
    ----------
    dge_df : pd.DataFrame
        DGE results. Can be bulk or single-cell.
        If multiple rows exist per gene (e.g. assays), a gene is considered
        significant if it is significant in ANY row.

    eqtl_sig_df : pd.DataFrame
        Significant eGenes only.

    eqtl_all_df : pd.DataFrame
        All tested eQTL genes.

    gene_col : str
        Gene symbol column in dge_df.

    sig_col : str
        Boolean significance column.

    eqtl_gene_col : str
        Gene symbol column in eqtl_sig_df.

    eqtl_all_gene_col : str
        Gene symbol column in eqtl_all_df.

    alternative : {"greater", "less", "two-sided"}

    Returns
    -------
    results : dict
        Contains odds ratio, p-value, contingency table, and counts.
    """

    # All genes tested in DGE
    dge_all = set(dge_df[gene_col].dropna())

    # Collapse to gene level:
    # significant if significant in ANY assay/cell type
    dge_sig = set(
        dge_df.loc[dge_df[sig_col], gene_col].dropna()
    )

    # eQTL genes
    eqtl_all = set(eqtl_all_df[eqtl_all_gene_col].dropna())
    eqtl_sig = set(eqtl_sig_df[eqtl_gene_col].dropna())

    # Restrict to genes tested in both analyses
    background = dge_all & eqtl_all

    dge_sig &= background
    eqtl_sig &= background

    # Contingency counts
    a = len(dge_sig & eqtl_sig)
    b = len(dge_sig - eqtl_sig)
    c = len(eqtl_sig - dge_sig)
    d = len(background - (dge_sig | eqtl_sig))

    contingency = pd.DataFrame(
        [[a, b],
         [c, d]],
        index=["eGene significant", "eGene not significant"],
        columns=["DGE significant", "DGE not significant"]
    )

    odds_ratio, pvalue = fisher_exact(
        [[a, b],
         [c, d]],
        alternative=alternative
    )

    return {
        "odds_ratio": odds_ratio,
        "pvalue": pvalue,
        "contingency": contingency,
        "background_n": len(background),
        "dge_sig_n": len(dge_sig),
        "eqtl_sig_n": len(eqtl_sig),
        "overlap_n": a,
    }

In [87]:
res = test_eqtl_enrichment(dge_idiopathic, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                   18                 4698
eGene not significant               36                 7224
Odds-Ratio: 0.768837803320562, P-Value: 0.8536410764127375


In [82]:
res = test_eqtl_enrichment(dge_22q11, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    4                  956
eGene not significant               63                12646
Odds-Ratio: 0.8398751411303712, P-Value: 0.7015899988899956


In [83]:
res = test_eqtl_enrichment(dge_nrxn1, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    6                 1354
eGene not significant               63                12525
Odds-Ratio: 0.8809875501160582, P-Value: 0.6760167469352049


In [84]:
res = test_eqtl_enrichment(dge_3q29, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    1                   22
eGene not significant               64                12582
Odds-Ratio: 8.936079545454545, P-Value: 0.11166952284155046


In [85]:
res = test_eqtl_enrichment(dge_15q13, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    0                    0
eGene not significant               60                12419
Odds-Ratio: nan, P-Value: 1.0


In [88]:
res = test_eqtl_enrichment(dge_idiopathic_bulk, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    7                  953
eGene not significant               50                11846
Odds-Ratio: 1.740230849947534, P-Value: 0.13052415919711705


In [89]:
res = test_eqtl_enrichment(dge_22q11_bulk, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    3                  156
eGene not significant               69                14808
Odds-Ratio: 4.127090301003345, P-Value: 0.040736746131051986


In [90]:
res = test_eqtl_enrichment(dge_nrxn1_bulk, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    1                  241
eGene not significant               73                15029
Odds-Ratio: 0.8542602171318138, P-Value: 0.692485335133914


In [91]:
res = test_eqtl_enrichment(dge_3q29_bulk, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    0                    0
eGene not significant               70                14089
Odds-Ratio: nan, P-Value: 1.0


In [92]:
res = test_eqtl_enrichment(dge_15q13_bulk, fetal_eqtl_df, fetal_all_eqtl_df)

print(res["contingency"])
print(f"Odds-Ratio: {res["odds_ratio"]}, P-Value: {res["pvalue"]}")

                       DGE significant  DGE not significant
eGene significant                    0                    0
eGene not significant               66                13746
Odds-Ratio: nan, P-Value: 1.0


In [71]:
## Idiopathic Bulk vs Fetal EQTLs 

# Gene sets

# All genes tested in the DGE analysis
dge_all = set(dge_idiopathic_bulk["ID"])

# Significant DGE genes
dge_sig = set(
    dge_idiopathic_bulk.loc[
        dge_idiopathic_bulk["significant_fdr"],
        "ID"
    ]
)

# All genes tested in the eQTL analysis
eqtl_all = set(fetal_all_eqtl_df["hgnc_symbol"])

# Significant eGenes (your fetal_eqtl_df)
eqtl_sig = set(fetal_eqtl_df["SYMBOL"])

# -----------------------------
# Restrict to common tested genes
# -----------------------------

background = dge_all & eqtl_all

dge_sig = dge_sig & background
eqtl_sig = eqtl_sig & background

# -----------------------------
# Build contingency table
# -----------------------------

a = len(dge_sig & eqtl_sig)                     # DGE sig & eGene sig
b = len(dge_sig - eqtl_sig)                    # DGE sig only
c = len(eqtl_sig - dge_sig)                    # eGene sig only
d = len(background - (dge_sig | eqtl_sig))     # neither

contingency = pd.DataFrame(
    [[a, b],
     [c, d]],
    index=["eGene significant", "eGene not significant"],
    columns=["DGE significant", "DGE not significant"]
)

print(contingency)

# -----------------------------
# Fisher exact test
# -----------------------------

odds_ratio, pvalue = fisher_exact(
    [[a, b],
     [c, d]],
    alternative="greater"   # enrichment
)

print(f"Odds ratio = {odds_ratio:.3f}")
print(f"P-value = {pvalue:.3e}")

                       DGE significant  DGE not significant
eGene significant                    7                  953
eGene not significant               50                11846
Odds ratio = 1.740
P-value = 1.305e-01


In [111]:
import pandas as pd
from scipy.stats import fisher_exact


def fisher_dge_eqtl(
    dge_df,
    eqtl_dict,
    dge_sig_col="significant_fdr",
    dge_gene_col="ID",
    eqtl_gene_col="hgnc_symbol",
    collapse_celltypes=True,
    background=None
):
    """
    Fisher exact enrichment of DGE significant genes among eQTL genes.

    Parameters
    ----------
    dge_df :
        DGE dataframe.
        Required columns:
            ID
            significant_fdr
            optionally assay

    eqtl_dict :
        Dictionary:
            {"celltype": dataframe}
        where dataframe contains eqtl_gene_col

    collapse_celltypes :
        If True:
            DGE significant if significant in ANY assay.
        If False:
            run each DGE assay separately.

    background :
        Optional universe of genes.
        If None uses union of DGE genes and eQTL genes.

    Returns
    -------
    dataframe
        Fisher results.
    """

    results = []

    # -----------------------
    # define DGE sets
    # -----------------------

    if "assay" in dge_df.columns and not collapse_celltypes:

        dge_sets = {
            ct: set(
                x.loc[x[dge_sig_col], dge_gene_col]
            )
            for ct, x in dge_df.groupby("assay")
        }

    else:

        dge_sets = {
            "all_DGE": set(
                dge_df.loc[
                    dge_df[dge_sig_col],
                    dge_gene_col
                ]
            )
        }


    # -----------------------
    # loop eQTL cell types
    # -----------------------

    for eqtl_name, eqtl_df in eqtl_dict.items():

        eqtl_genes = set(
            eqtl_df[eqtl_gene_col]
            .dropna()
            .unique()
        )


        for dge_name, dge_genes in dge_sets.items():

            if background is None:

                universe = (
                    set(dge_df[dge_gene_col])
                    |
                    eqtl_genes
                )

            else:
                universe = set(background)


            dge_genes = dge_genes & universe
            eqtl_genes = eqtl_genes & universe


            # contingency table
            #
            #              eQTL  no eQTL
            # DGE          a      b
            # no DGE       c      d

            a = len(dge_genes & eqtl_genes)

            b = len(dge_genes - eqtl_genes)

            c = len(eqtl_genes - dge_genes)

            d = len(
                universe
                -
                dge_genes
                -
                eqtl_genes
            )


            odds, p = fisher_exact(
                [[a,b],
                 [c,d]],
                alternative="greater"
            )


            results.append({

                "DGE_group": dge_name,
                "eQTL_celltype": eqtl_name,

                "DGE_sig_genes": len(dge_genes),
                "eQTL_genes": len(eqtl_genes),

                "overlap": a,

                "odds_ratio": odds,

                "p_value": p

            })


    out = pd.DataFrame(results)

    out["FDR"] = (
        out["p_value"]
        .rank(method="min")
        /
        len(out)
    )

    return out.sort_values("p_value")

In [112]:
singlebrain_eqtls = {

    "OPC": opc_singlebrain_eqtl,
    "IN": in_singlebrain_eqtl,
    "MG": mg_singlebrain_eqtl,
    "OD": od_singlebrain_eqtl,
    "End": end_singlebrain_eqtl

}

In [113]:
bulk_results = fisher_dge_eqtl(
    dge_idiopathic_bulk,
    singlebrain_eqtls
)

bulk_results

,DGE_group,eQTL_celltype,DGE_sig_genes,eQTL_genes,overlap,odds_ratio,p_value,FDR
0,all_DGE,OPC,1115,11960,684,0.450797,1.0,0.2
1,all_DGE,IN,1115,12163,691,0.447774,1.0,0.2
2,all_DGE,MG,1115,12261,684,0.425916,1.0,0.2
3,all_DGE,OD,1115,11449,639,0.439117,1.0,0.2
4,all_DGE,End,1115,15030,847,0.428286,1.0,0.2


In [114]:
fetal_cell_results = fisher_dge_eqtl(
    dge_idiopathic,
    singlebrain_eqtls,
    collapse_celltypes=True
)

fetal_cell_results

,DGE_group,eQTL_celltype,DGE_sig_genes,eQTL_genes,overlap,odds_ratio,p_value,FDR
0,all_DGE,OPC,5081,11960,3949,0.792983,1.0,0.2
4,all_DGE,End,5081,15030,4453,0.742797,1.0,0.4
1,all_DGE,IN,5081,12163,3956,0.781527,1.0,0.6
2,all_DGE,MG,5081,12261,3960,0.740897,1.0,0.8
3,all_DGE,OD,5081,11449,3773,0.741808,1.0,1.0


In [115]:
fetal_by_cell_results = fisher_dge_eqtl(
    dge_idiopathic,
    singlebrain_eqtls,
    collapse_celltypes=False
)

fetal_by_cell_results

,DGE_group,eQTL_celltype,DGE_sig_genes,eQTL_genes,overlap,odds_ratio,p_value,FDR
12,Radial Glia SOX2+PAX6+FABP7+,IN,1855,12163,1586,1.493906,1.228182e-09,0.028571
5,Radial Glia SOX2+PAX6+FABP7+,OPC,1855,11960,1573,1.434377,2.902110e-08,0.057143
19,Radial Glia SOX2+PAX6+FABP7+,MG,1855,12261,1571,1.334022,8.408580e-06,0.085714
26,Radial Glia SOX2+PAX6+FABP7+,OD,1855,11449,1511,1.298550,1.477451e-05,0.114286
33,Radial Glia SOX2+PAX6+FABP7+,End,1855,15030,1710,1.408621,5.067637e-05,0.142857
1,Hindbrain Neurons NR2F2+PBX3+LHX1+,OPC,83,11960,76,2.691446,3.845617e-03,0.171429
20,Radial Glia SOX2+VIM+FABP7+,MG,46,12261,44,5.150201,4.344860e-03,0.200000
22,Hindbrain Neurons NR2F2+PBX3+LHX1+,OD,83,11449,73,2.099648,1.306572e-02,0.228571
27,Radial Glia SOX2+VIM+FABP7+,OD,46,11449,42,3.017358,1.425637e-02,0.257143
15,Hindbrain Neurons NR2F2+PBX3+LHX1+,MG,83,12261,75,2.195655,1.601148e-02,0.285714


In [116]:
singlebrain_sig = {}

for ct, df in singlebrain_eqtls.items():

    sig = df[df["qval"] < 0.05].copy()

    singlebrain_sig[ct] = sig

    print(
        ct,
        len(sig["hgnc_symbol"].unique())
    )

OPC 9286
IN 11117
MG 8387
OD 9853
End 4231


In [117]:
sig_results = fisher_dge_eqtl(
    dge_idiopathic,
    singlebrain_sig,
    collapse_celltypes=False
)

sig_results.head(20)

,DGE_group,eQTL_celltype,DGE_sig_genes,eQTL_genes,overlap,odds_ratio,p_value,FDR
12,Radial Glia SOX2+PAX6+FABP7+,IN,1855,11116,1458,1.286797,0.000011,0.028571
5,Radial Glia SOX2+PAX6+FABP7+,OPC,1855,9285,1229,1.130775,0.009989,0.057143
26,Radial Glia SOX2+PAX6+FABP7+,OD,1855,9852,1308,1.124805,0.015957,0.085714
22,Hindbrain Neurons NR2F2+PBX3+LHX1+,OD,83,9852,65,1.678076,0.029719,0.114286
8,Hindbrain Neurons NR2F2+PBX3+LHX1+,IN,83,11116,68,1.545305,0.076040,0.142857
27,Radial Glia SOX2+VIM+FABP7+,OD,46,9852,36,1.670905,0.095514,0.171429
20,Radial Glia SOX2+VIM+FABP7+,MG,46,8386,31,1.507640,0.121684,0.200000
24,Mixed Neurons FGF12+GRIN2B+CAMK2B+,OD,5,9852,5,inf,0.148921,0.228571
6,Radial Glia SOX2+VIM+FABP7+,OPC,46,9285,33,1.441260,0.166949,0.257143
14,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,MG,76,8386,48,1.250454,0.205893,0.285714


In [129]:
import pandas as pd
from scipy.stats import spearmanr


def prepare_eqtl_lead(eqtl_df):
    """
    Keep the lead eQTL per gene (lowest nominal p-value)
    """

    eqtl = (
        eqtl_df
        .sort_values("pval_nominal")
        .drop_duplicates("hgnc_symbol")
        [["hgnc_symbol", "slope", "pval_nominal"]]
        .copy()
    )

    return eqtl


def correlate_dge_eqtl(
    dge_df,
    eqtl_df,
    dge_gene_col="ID",
    dge_effect_col="estimate",
):

    # collapse DGE to gene level
    dge = (
        dge_df
        .groupby(dge_gene_col)[dge_effect_col]
        .mean()
        .reset_index()
    )

    # merge
    merged = dge.merge(
        eqtl_df,
        left_on=dge_gene_col,
        right_on="hgnc_symbol"
    )

    merged = merged.dropna()

    if len(merged) < 20:
        return {
            "n_genes": len(merged),
            "rho": None,
            "p": None
        }

    rho, p = spearmanr(
        merged[dge_effect_col],
        merged["slope"]
    )

    return {
        "n_genes": len(merged),
        "rho": rho,
        "p": p
    }

In [130]:
fetal_lead_eqtl = prepare_eqtl_lead(fetal_all_eqtl_df)

fetal_lead_eqtl.head()

,hgnc_symbol,slope,pval_nominal
50527165,CUTALP,-1.13028,9.857990e-50
64757173,TAS2R43,-1.16221,1.237310e-40
98578776,MTCO2P2,-1.16866,4.732140e-40
113978062,NaN,1.17451,1.368230e-38
94334719,MAPK8IP1P2,1.40980,5.061930e-36


In [131]:
results = []

for ct in dge_idiopathic["assay"].unique():

    dge_ct = dge_idiopathic[
        dge_idiopathic["assay"] == ct
    ]

    res = correlate_dge_eqtl(
        dge_ct,
        fetal_lead_eqtl
    )

    res["celltype"] = ct
    results.append(res)


fetal_eqtl_correlations = pd.DataFrame(results)

fetal_eqtl_correlations

,n_genes,rho,p,celltype
0,11913,0.013509,0.140393,Mesenchymal-like cells VIM+VCAN+SPARC+
1,8883,0.000075,0.994325,Proliferative Radial Glia SOX2+HES6+TOP2A+
2,8210,0.016476,0.135496,Radial Glia SOX2+PAX6+FABP7+
3,2845,0.005594,0.765534,Radial Glia SOX2+VIM+FABP7+
4,511,0.038850,0.380809,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+
5,793,0.013196,0.710606,Hindbrain Neurons NR2F2+PBX3+LHX1+
6,8,NaN,NaN,Mixed Neurons FGF12+GRIN2B+CAMK2B+


In [133]:
dge_idiopathic_bulk['significant_fdr'].sum()

1115